In [3]:
!pip install -q -U transformers peft accelerate scikit-learn pillow


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoImageProcessor,
    SiglipForImageClassification
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Imports successful.")

Imports successful.


In [4]:
SEED = 42

MODEL_NAME = "google/siglip-base-patch16-224"

CLASS_NAMES = [
    "Red sandstone",
    "Light sandstone",
    "Gray siltstone",
    "Mudstone",
    "Granite",
    "Basalt",
    "Marble"
]

LABEL2ID = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

ID2LABEL = {
    i: name
    for i, name in enumerate(CLASS_NAMES)
}

NUM_CLASSES = len(CLASS_NAMES)

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training configuration
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Model:", MODEL_NAME)
print("Classes:", NUM_CLASSES)
print("LoRA rank:", LORA_R)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

Model: google/siglip-base-patch16-224
Classes: 7
LoRA rank: 16
Epochs: 3
Batch size: 16


In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("WARNING: No GPU detected.")


Device: cpu


In [4]:
from pathlib import Path

# Automatically detect whether we're running locally or on Kaggle
if Path("/kaggle/working").exists():
    PROJECT_ROOT = Path("/kaggle/working/GeoSigLIP")
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_FILE = DATA_DIR / "train.csv"
VAL_FILE = DATA_DIR / "validation.csv"

print("Project:", PROJECT_ROOT)
print("Train file:", TRAIN_FILE)
print("Validation file:", VAL_FILE)

Project: C:\Documents\GeoSigLIP
Train file: C:\Documents\GeoSigLIP\data\processed\train.csv
Validation file: C:\Documents\GeoSigLIP\data\processed\validation.csv


In [5]:
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

Train: 2450
Validation: 1050

Train distribution:
label
Granite            350
Red sandstone      350
Gray siltstone     350
Mudstone           350
Basalt             350
Marble             350
Light sandstone    350
Name: count, dtype: int64

Validation distribution:
label
Basalt             150
Marble             150
Mudstone           150
Granite            150
Light sandstone    150
Red sandstone      150
Gray siltstone     150
Name: count, dtype: int64


In [6]:
train_df["label_id"] = train_df["label"].map(LABEL2ID)
val_df["label_id"] = val_df["label"].map(LABEL2ID)

print(train_df[["label", "label_id"]].drop_duplicates())

              label  label_id
0           Granite         4
1     Red sandstone         0
2    Gray siltstone         2
4          Mudstone         3
6            Basalt         5
7            Marble         6
18  Light sandstone         1


In [8]:
%pip install -U torchvision

   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.2 MB 6.7 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.2 MB 6.7 MB/s eta 0:00:01
   ----------------------------------- ---- 3.7/4.2 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%pip install --force-reinstall torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/110.9 MB 1.6 MB/s eta 0:01:10
   ---------------------------------------- 1.0/110.9 MB 1.7 MB/s eta 0:01:04
    --------------------------------------- 1.8/110.9 MB 2.1 MB/s eta 0:00:53
    --------------------------------------- 2.4/110.9 MB 2.3 MB/s eta 0:00:49
   - ----------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
